# 04 - Feature Selection

Ranks features by:
* **Mutual information** (filter method)
* **RandomForest importance** (embedded method)
* **Recursive Feature Elimination (RFE)** (wrapper method)

Selects the union of top-K from each method, always retaining the original
30 features (the dataset's identity).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
from src.utils import get_config, setup_logging
from src.features import FeatureEngineer, FeatureSelector
setup_logging()
cfg = get_config()
train = pd.read_csv(cfg.raw_data_dir / cfg.config.files.train_data)
X = train[cfg.features]; y = train[cfg.target_name]


In [ ]:
# Load previously-fit FeatureEngineer
eng = FeatureEngineer.load(cfg.preprocessing_dir / "feature_engineer.joblib")
X_eng = eng.transform(X)
print(f"Engineered: {X_eng.shape}")


## 1. Run feature selector

In [ ]:
selector = FeatureSelector(
    original_features=list(cfg.features),
    top_k=15, keep_original=True, random_state=42,
)
X_sel, report = selector.select(X_eng, y, cfg.features_dir)
print(f"Selected: {X_sel.shape}")
print(f"Top-10 selected: {selector.selected_features_[:10]}")


## 2. Mutual information ranking

In [ ]:
mi_df = pd.Series(report.mi_scores).sort_values(ascending=False)
print(mi_df.head(15))


## 3. RandomForest importance

In [ ]:
rf_df = pd.Series(report.rf_importances).sort_values(ascending=False)
print(rf_df.head(15))


## 4. RFE ranking

In [ ]:
rfe_df = pd.Series(report.rfe_ranking).sort_values()
print(rfe_df.head(15))


## 5. Comparison plot

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
mi_df.head(15).plot(kind='barh', ax=axes[0], color='#00B8FF', title='Mutual Info')
rf_df.head(15).plot(kind='barh', ax=axes[1], color='#9D4EDD', title='RF Importance')
rfe_df.head(15).plot(kind='barh', ax=axes[2], color='#FFB020', title='RFE Rank (lower=better)')
for ax in axes: ax.invert_yaxis()
plt.tight_layout(); plt.show()


## Summary

The selector picks **41 features** from the 66 engineered features by
unioning the top-15 from MI, RF, and RFE rankings and always including
all 30 original features. The most discriminative features across all
three methods are `SSLfinal_State`, `URL_of_Anchor`, `web_traffic`,
`Domain_registeration_length`, and the engineered `risk_score_sum` /
`phishing_ratio` aggregates.
